## A workshop for developing weights for match ratings algorithms

For now, we will look at using the PCA algorithm

We will start with just looking at one position: CM

### Loading the data
First, we need to load the full matches dataset, defining the root folder and the path to the matches dataset.

In [224]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 93 matches from C:\Users\kazik\projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


### Filtering the data
Now that we have the full matches dataset loaded, we can collect it into a pandas DataFrame.
In doing so, we will focus it on the player_performances section of the dataset, discarding the rest of the data (match metadata, teams, team stats, etc.), with the exception of the match_id, which we will keep as a reference to the match in which the performance took place, the half_length field, which we will use to normalize the performance stats, and the main team's xG and possession. We will then filter to only include performances from the CM position.

In [225]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_stats", "passes"],
        ["data", "away_stats", "passes"]
    ]
)
# Create 4 new columns: team_possession, team_xg, opponent_xg, and team_passes, and set all the values to NaN
normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")
normalized_df["team_passes"] = float("nan")

for index, performance in normalized_df.iterrows():
    # Check if the performance is for the home team or away team
    if performance["data.home_team_name"] == "Valencia CF":
        normalized_df.at[index, "team_possession"] = performance["data.home_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.home_stats.passes"]
    else:
        normalized_df.at[index, "team_possession"] = performance["data.away_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.away_stats.passes"]

# Remove the home and away names, possession, xg, and passes columns
normalized_df = normalized_df.drop(columns=["data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg", "data.home_stats.passes", "data.away_stats.passes"])

normalized_df = normalized_df.rename(columns={"id": "match_id"})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

# Now we want to filter for players with only "CM" in "positions_played"
cm_players_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["CM"])]

# Now remove columns that are all NaN
cm_players_df = cm_players_df.dropna(axis=1, how="all")

# Remove "performance_type" and "positions_played" columns
cm_players_df = cm_players_df.drop(columns=["performance_type", "positions_played"])

# Move "match_id" to the front, and rename the id column to "performance_id"
cols = cm_players_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
cm_players_df = cm_players_df[cols]
cm_players_df.head()

# Rename "data.half_length" to "half_length"
cm_players_df = cm_players_df.rename(columns={"data.half_length": "half_length"})

# Output number of rows and columns, and the first few rows of the resulting dataframe
print(f"Number of rows: {cm_players_df.shape[0]}")
print(f"Number of columns: {cm_players_df.shape[1]}")
print("Columns:")
print(cm_players_df.columns.tolist())
print("\n")

# Output the max and min of every column, to get a sense of the range of values
for col in cm_players_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if cm_players_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={cm_players_df[col].min()}, max={cm_players_df[col].max()}")

Number of rows: 251
Number of columns: 24
Columns:
['match_id', 'player_id', 'goals', 'assists', 'shots', 'shot_accuracy', 'passes', 'pass_accuracy', 'dribbles', 'dribble_success_rate', 'tackles', 'tackle_success_rate', 'offsides', 'fouls_committed', 'possession_won', 'possession_lost', 'minutes_played', 'distance_covered', 'distance_sprinted', 'half_length', 'team_possession', 'team_xg', 'opponent_xg', 'team_passes']


goals: min=0.0, max=8.0
assists: min=0.0, max=8.0
shots: min=0.0, max=12.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=55.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=54.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=12.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=8.0
fouls_committed: min=0.0, max=8.0
possession_won: min=0.0, max=8.0
possession_lost: min=0.0, max=14.0
minutes_played: min=2.0, max=94.0
distance_covered: min=0.3, max=14.1
distance_sprinted: min=0.1, max=8.0
team_possession: min=39.0, max=71.0


### Fixing data types
Now that we have the data in a DataFrame, we need to fix the data types of the columns. We will convert all the columns that are not identifiers (like "match_id" and "player_id") to numeric data types. This is especially important when it comes to calculating xT, as we will need to perform mathematical operations on the stats.

In [226]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = cm_players_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].fillna(0)

### Data pre-processing and normalizing
Now that we have the data in a DataFrame, we can start pre-processing it. A number of pre-processing steps are needed before we can apply PCA to the data.<br>

**Minutes played minimum**<br>
First, we will filter the data to only include performances where the player played at least 10 minutes. This is to ensure that we are only looking at performances where the player had a significant impact on the match, and to avoid skewing the PCA results with performances where the player only played a few minutes.<br>


In [227]:
# Step 1: Removing players with less than 10 minutes played
cm_players_df = cm_players_df[cm_players_df["minutes_played"] >= 10]

**Volume masks**<br>
We need to implement a volume mask. For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage stat to be considered valid. If the volume attempted stat does not reach the threshold, we will replace the percentage stat with the mean percentage stat for that stat across all performances that do reach the threshold. This is to ensure that we are not skewing the PCA results with invalid percentage stats. The thresholds for the masks will be as follows: Pass accuracy - 5 attempted passes, Shot accuracy - 2 attempted shots, Dribble success rate - 3 attempted dribbles, Tackle success rate - 2 attempted tackles.<br>

In [228]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = cm_players_df[cm_players_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    cm_players_df[perc_col] = cm_players_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

**Possession adjustment**<br>
One problem with this dataset is that it is for a top top team, one that in general has a lot of possession, and the stats will be skewed by this. This means that with a top team set as the average, what would normally be a very good performance for a player on a weaker, lower possession team, may be seen as average or even bad, as he isn't producing as much as players on a top team who simply see more of the ball.<br>
Two sets of stats are adjusted to account for this: Volume stats that are positively influenced by their team's possession (passes attempted, dribbles attempted, shots attempted, possession lost) are sdjusted by multiplying them by:<br>$$PAdj_{Attacking} = Raw \times \left( \frac{50}{Team\_Possession} \right)$$<br>Volume stats that are negatively influenced by their team's possession (tackles, possession won, fouls committed) are adjusted by multiplying them by:<br>$$PAdj_{Defensive} = Raw \times \left( \frac{50}{100 - Team\_Possession} \right)$$<br>

In [229]:
# Passes mean and std before adjustment
print(f"Passes mean before adjustment: {cm_players_df['passes'].mean()}, std: {cm_players_df['passes'].std()}")

# Possession adjustment
# 1. The Softened Attacking & Distribution Tax
atk_cols = ["passes", "dribbles", "shots", "possession_lost"]
for col in atk_cols:
    # np.sqrt softens the penalty so extreme performances aren't crushed
    cm_players_df[col] = cm_players_df[col] * np.sqrt(50.0 / cm_players_df["team_possession"])

# 2. The Softened Defensive Tax
def_cols = ["tackles", "possession_won", "fouls_committed"]
for col in def_cols:
    cm_players_df[col] = cm_players_df[col] * np.sqrt(50.0 / (100.0 - cm_players_df["team_possession"]))
# print the passes mean and std after adjustment
print(f"Passes mean after adjustment: {cm_players_df['passes'].mean()}, std: {cm_players_df['passes'].std()}")

Passes mean before adjustment: 24.50409836065574, std: 13.83545215093233
Passes mean after adjustment: 23.20528861052775, std: 12.710648190255892


**Positional xG share**<br>
Two volume stats that were not adjusted by possession are goals and assists. But this means that all strikers will be judged based on the performance of some of the best strikers, who could well be recording something like 0.9 goals per 90. This means that a striker who scores 0.5 goals per 90, which would be an excellent return for a striker on a mid-table team, may be seen as a poor return compared to the top strikers. The solution for this is positional xG share. Based on this inital dataset, we calculate what percentage of the team's goals are scored by each position. For a striker this could, for example, be 45%. Then, when calculating the z-scores for each performance, instead of using a static mean for goals, we calculate a dynamic mean based on the service the team provided:<br>$$\mu_{Goals} = Team\_Match\_xG \times Positional\_Share$$<br>This means that for a top team, the team may dominate and create 3.5xG, making the striker's dynamic mean 0.45. Therefore, if the striker scores 0 goals, their z-score will show a massive underperformance, heavily penalizing them for wasting the great service they were given. On the other hand, in a team fighting relegation, the team may only record 0.4xG in a match, making the striker's dynamic mean 0.18. Therefore, if the striker scores 0 goals, their z-score will show a much smaller underperformance, as they were not given much in the way of chances to score in the first place, and they are not being heavily penalized for failing to score when they were barely given any opportunity to do so. The exact same logic applies to assists, calculating the dynamic mean based on the team's xG and the positional share of assists, and then calculating the z-score based on that dynamic mean.<br>

In [230]:
# find cm goal share, using the original normalized_df, going thru every performance
# and keeping a running total of both the total number of goals scored by every player
# and the total number of goals scored by cm players, and then dividing the two at the end to get the goal share of cm players
total_goals = 0
cm_goals = 0

total_assists = 0
cm_assists = 0

for index, performance in normalized_df.iterrows():
    if performance["performance_type"] == "GK":
        continue  # Skip goalkeepers for this check
    total_goals += performance["goals"]
    total_assists += performance["assists"]
    if performance["positions_played"] == ["CM"]:
        cm_goals += performance["goals"]
        cm_assists += performance["assists"]

cm_goal_share = cm_goals / total_goals if total_goals > 0 else 0
print(f"CM Goal Share: {cm_goal_share:.2%}")

cm_assist_share = cm_assists / total_assists if total_assists > 0 else 0
print(f"CM Assist Share: {cm_assist_share:.2%}")

# Overriding this to hardcode goal share at 10% and assist share at 15%, to give them more weight in the final rating
cm_goal_share = 0.1
cm_assist_share = 0.15

CM Goal Share: 25.56%
CM Assist Share: 34.17%


**Normalizing by half length and minutes played**<br>
All volume stats will be normalized by two factors: the IRL length of the half, as determined by the user-set EA FC game settings, and the minutes played by the player in that performance. This it to ensure players are valued for their per-minute impact on the game (similarly how a player who scores 1 goal in 90 minutes is less impressive than a player who scores 1 goal in 10 minutes), and to account for the fact that the half length in EA FC can be set to different lengths, and we want our weights to be applicable regardless of the half length setting. Players should not have a lower rating purely because the user has less spare time to play FIFA, and if they increase the half length part way through the career, their player's ratings should not suddenly jump as a result of having more time to make passes etc. This is done using the following formula:
$$X_{final} = X_{raw} \times \left( \frac{90}{M_{played}} \right) \times \left( \frac{H_{base}}{H_{match}} \right)$$
where:<br>
 - $X_{final}$ is the final, normalized stat value that will be used in the PCA<br>
 - $X_{raw}$ is the raw, unnormalized stat value from the dataset<br>
 - $M_{played}$ is the minutes played by the player in that performance<br>
 - $H_{base}$ is the base half length, which we will set to 10<br>
 - $H_{match}$ is the half length for that match, as determined by the half_length field in the dataset<br>

In [231]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    cm_players_df[col] = cm_players_df[col] * (h_base / cm_players_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    cm_players_df[f"{col}_p90"] = (cm_players_df[col] / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (cm_players_df[col].sum() / cm_players_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    cm_players_df[f"{col}_p90"] = ((cm_players_df[col] + dummy_stat) / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0

# Print to verify
print(f"Passes mean after smoothing: {cm_players_df['passes_p90'].mean():.2f}, std: {cm_players_df['passes_p90'].std():.2f}")

Passes mean after smoothing: 33.21, std: 5.70


**Expected Threat**<br>
Expected Threat (xT) is a metric that measures the likelihood of a player scoring or assisting a goal based on their actions on the pitch.<br>True Expected Threat (xT) models operate by laying a 'value surface' over the pitch, dividing it into spatial zones assigned distinct probabilities of yielding a goal. Because exact XY spatial coordinate tracking is not available in the provided JSON structure, true Expected Threat (xT) surfaces cannot be explicitly drawn.<br>However, a highly correlated proxy for vertical progression and threat generation is engineered via the ratio of `distance_sprinted` to `distance_covered`, coupled with passing volume and efficiency. Modern physical analytics indicate that high-speed running, when paired with sustained ball retention and distribution, accounts for structural line-breaking and spatial disruption. A continuous bonus is added via NumPy broadcasting to mimic xT::
$$Bonus_{xT} = 0.25 \times \left( \frac{D_{sprinted}}{D_{covered}} \right) \times \ln(Pass_{acc} \times Passes + 1)$$
This proxy ensures players executing high-intensity runs combined with high passing volume receive the mathematical credit normally reserved for spatial xT models.

In [232]:
cm_players_df["xT_bonus_p90"] = 0.25 * (cm_players_df["distance_sprinted_p90"] / cm_players_df["distance_covered_p90"]) * np.log(cm_players_df["pass_accuracy"] * cm_players_df["passes_p90"] + 1)

**Handling long right tails**<br>
Z-scores, and therefore PCA results, expect a normal distribution, and can therefore be heavily skewed by long tails in either direction. In this dataset, we have a number of stats with long right tails, for example goals and assists, where the majority of performances will be clustered around 0, with a long tail of performances with 1 or more goals/assists. To account for this, we will apply a log transformation to all stats with long right tails, which will help to reduce the skewness of the distribution and make it more normal. The log transformation is applied using the following formula:
$$X_{log} = \log(X_{final} + 1)$$
where:<br>
 - $X_{log}$ is the log-transformed stat value that will be used in the PCA<br>
 - $X_{final}$ is the final, normalized stat value from the previous step<br>
 - We add 1 to $X_{final}$ to avoid taking the log of 0, which is undefined<br>

In [233]:
# Columns to log for PCA only; do NOT modify cm_players_df in-place
cols_to_log = ["goals_p90", "assists_p90", "shots_p90", "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90"]

**Z-score calculation**<br>
Finally, to turn the data into a format that can be used for PCA, we will calculate the z-scores for each stat. There are two different ways we will calculate the z-score, based on the stat in question:
 - For negative stats that we want to penalize (fouls committed, offsides, possession lost), we use an inverted z-score formula, where a higher raw stat value results in a lower z-score, as we want to penalize players for having high values in these stats. The formula for this is:<br>$$Z_{inverted} = \frac{\mu - X}{\sigma}$$
 - For all other stats, we use the standard z-score formula, where a higher raw stat value results in a higher z-score, as we want to reward players for having high values in these stats. The formula for this is:<br> $$Z = \frac{X - \mu}{\sigma}$$

In [234]:
negative_stats = ["fouls_committed_p90", "offsides_p90", "possession_lost_p90"]

# Build PCA dataframe from cm_players_df and apply log to long-tail columns (PCA-only)
pca_df = cm_players_df.copy()
pca_df[cols_to_log] = np.log(pca_df[cols_to_log] + 1)

# Define ordered stat list used for PCA (must match final weight order)
stat_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
# Calculate z-scores on the log-transformed PCA dataframe
for col in stat_names:
    if col == "goals":
        mean = np.log(cm_players_df["team_xg"] * cm_goal_share + 1)
        std = np.log(pca_df[col]).std()
    elif col == "assists":
        mean = np.log(cm_players_df["team_xg"] * cm_assist_share + 1)
        std = np.log(pca_df[col]).std()
    else:
        mean = pca_df[col].mean()
        std = pca_df[col].std()
    if std == 0:
        pca_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pca_df[f"{col}_z"] = (mean - pca_df[col]) / std
    else:
        pca_df[f"{col}_z"] = (pca_df[col] - mean) / std

# final matrix for PCA (z-scores from the log-transformed matrix)
final_cm_df = pca_df[[f"{col}_z" for col in stat_names]]

### Principal Component Analysis (PCA)
Now that we have pre-processed and normalized the data, we can apply PCA to it to identify the most important stats for evaluating player performance in the CM position, and to calculate the weights for each stat based on their contribution to the principal components. PCA will help us to reduce the dimensionality of the data while retaining as much of the variance as possible, allowing us to identify the key drivers of player performance and to create a more accurate and robust match rating algorithm. We are using PCA as it is a well-established and widely used technique for dimensionality reduction and feature extraction, and it will allow us to identify the underlying structure of the data and to create a more interpretable and actionable set of weights for our match rating algorithm.<br><br>
To handle the higher-than-usual variance, we will use a robust covariance estimator, the Minimum Covariance Determinant (MCD), which is designed to be robust to outliers and can help to improve the performance of PCA in the presence of high variance. The MCD will help us to identify the most representative subset of the data for calculating the covariance matrix, which is a key step in PCA, and will allow us to create a more accurate and reliable set of weights for our match rating algorithm.<br><br>
Testing showed that the support fraction of the MCD (the proportion of the data that is used to calculate the covariance matrix) needed to be 0.98, larger than expected, to ensure that the PCA results were stable and consistent across different runs. This is likely due to the fact that even after pre-processing, the data still contains some outliers and high variance, and using a larger support fraction allows the MCD to capture more of the data and to create a more accurate covariance matrix for PCA.<br><br>
We thene extract the eigenvalues and eigenvectors from the PCA, which will allow us to calculate the weights for each stat based on their contribution to the principal components. The eigenvalues represent the amount of variance explained by each principal component, while the eigenvectors represent the direction of the principal components in the original feature space. By analyzing the eigenvalues and eigenvectors, we can identify which stats are most important for evaluating player performance in the CM position, and we can calculate the weights for each stat based on their contribution to the principal components.

In [235]:
# 1. Define the "Blocks" of your Central Midfielder
blocks = {
    "Attacking": [
        "goals_p90_z", "assists_p90_z", "shots_p90_z", "shot_accuracy_z", "offsides_p90_z"
    ],
    "Passing": [
        "passes_p90_z", "pass_accuracy_z"
    ],
    "Dribbling": [
        "dribbles_p90_z", "dribble_success_rate_z"
    ],
    "Safety": [
        "possession_lost_p90_z",
    ],
    "Defending": [
        "tackles_p90_z", "tackle_success_rate_z", "possession_won_p90_z", "fouls_committed_p90_z"
    ],
    "Workrate_and_Threat": [
        "distance_covered_p90_z", "distance_sprinted_p90_z", "xT_bonus_p90_z"
    ]
}

# 2. Create a copy of your PCA dataframe so we don't mess up the original Z-scores
# Assume 'final_cm_df' is the dataframe containing just the 17 Z-score columns
scaled_pca_df = final_cm_df.copy()

# 3. Apply the Block Scaling math (Divide by the square root of the block size)
for stats in blocks.values():
    k = len(stats)  # Number of stats in this block
    scale_factor = np.sqrt(k)

    for stat in stats:
        # Scale the Z-score down
        scaled_pca_df[stat] = scaled_pca_df[stat] / scale_factor

# 4. Now run your robust PCA on the SCALED dataframe
mincovdet_random_state = 42

mcd = MinCovDet(random_state=mincovdet_random_state, support_fraction=0.98).fit(scaled_pca_df)

cov = mcd.covariance_

eigenvalues, eigenvectors = np.linalg.eigh(cov)

sorted_indicies = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indicies]
eigenvectors = eigenvectors[:, sorted_indicies]

k = 9

top_eigenvectors = eigenvectors[:, :k]
top_variances = eigenvalues[:k]

weighted_loadings = np.abs(top_eigenvectors) * np.sqrt(top_variances)

raw_weights = np.sum(weighted_loadings, axis=1)

# Normalize to sum to 1
final_weights = raw_weights / np.sum(raw_weights)

stat_names = ["goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy",
              "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides",
              "fouls_committed", "possession_won", "possession_lost", "distance_covered", "distance_sprinted", "xT_bonus"]

print("\nFinal weights (normalized to sum to 1):")
for stat, weight in zip(stat_names, final_weights):
    # print to 4 dp
    print(f"{stat}: {weight:.4f}")


Final weights (normalized to sum to 1):
goals: 0.0434
assists: 0.0421
shots: 0.0434
shot_accuracy: 0.0330
passes: 0.0981
pass_accuracy: 0.0855
dribbles: 0.0686
dribble_success_rate: 0.0807
tackles: 0.0582
tackle_success_rate: 0.0606
offsides: 0.0006
fouls_committed: 0.0567
possession_won: 0.0603
possession_lost: 0.0895
distance_covered: 0.0514
distance_sprinted: 0.0643
xT_bonus: 0.0636


In [236]:
# mincovdet_random_state = 42

# mcd = MinCovDet(random_state=mincovdet_random_state, support_fraction=0.98).fit(final_cm_df)

# cov = mcd.covariance_

# eigenvalues, eigenvectors = np.linalg.eigh(cov)

# sorted_indicies = np.argsort(eigenvalues)[::-1]
# eigenvalues = eigenvalues[sorted_indicies]
# eigenvectors = eigenvectors[:, sorted_indicies]

### Analysing the PCA results and calculating the weights
Now that we have the PCA results, we can analyze them to identify the most important stats for evaluating player performance in the CM position, and to calculate the weights for each stat based on their contribution to the principal components.<br><br>
First, we calculate a list of the explained variance ratios for each principal component, which will allow us to understand how much of the variance in the data is explained by each component. We can then calculate the cumulative explained variance to determine how many principal components we need to retain in order to capture a certain percentage of the variance in the data, for example 85%. 85% was chosen as the threshold for retaining principal components as it is a commonly used threshold in PCA that balances the trade-off between retaining enough variance in the data and reducing the dimensionality of the data to a manageable level, and avoiding retaining componenents that may be capturing statistical noise rather than meaningful patterns in the data.<br><br>
Now we extract the top $k$ principal compenents (where $k$ is the number of components needed to reach the 85% explained variance threshold). We multiply the eigenvectors for these top $k$ components by their corresponding eigenvalues to calculate the contribution of each original stat to the principal components. We then find the raw weights, which are the sum of the contributions of each stat across the top $k$ principal components. Finally, we normalize the raw weights to a scale of 0 to 1 to create the final weights for each stat, which can be used in our match rating algorithm to evaluate player performance in the CM position. The stats with the highest weights will be the most important for evaluating player performance, while the stats with lower weights will be less important.

In [237]:
# explained_variance = (eigenvalues / np.sum(eigenvalues)) * 100
# # Show them as percentages with 2 decimal places
# explained_variance = [f"{ev:.2f}%" for ev in explained_variance]
# print("Explained Variance by each principal component:")
# for i, ev in enumerate(explained_variance):
#     print(f"Principal Component {i+1}: {ev}")

# # Now we want to find the number of principal components needed to explain at least 85% of the variance
# cumulative_variance = np.cumsum([float(ev.strip("%")) for ev in explained_variance])
# sum_to_85 = np.argmax(cumulative_variance >= 85) + 1
# print(f'The number of principal components needed to explain at least 85% of the variance is: {sum_to_85}\n')

# k = 9

# top_eigenvectors = eigenvectors[:, :k]
# top_variances = eigenvalues[:k]

# weighted_loadings = np.abs(top_eigenvectors) * np.sqrt(top_variances)

# raw_weights = np.sum(weighted_loadings, axis=1)

# # Normalize to sum to 1
# final_weights = raw_weights / np.sum(raw_weights)

# stat_names = ["goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy",
#               "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides",
#               "fouls_committed", "possession_won", "possession_lost", "distance_covered", "distance_sprinted", "xT_bonus"]

# print("\nFinal weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, final_weights):
#     # print to 4 dp
#     print(f"{stat}: {weight:.4f}")

# # Throw away the temporary PCA/log matrix to avoid accidentally using logged data later
# del pca_df

In [238]:
# # print the full weights from component 1 and component 2
# comp_1_weights = np.abs(eigenvectors[:, 0]) * eigenvalues[0]
# comp_1_weights = comp_1_weights / np.sum(comp_1_weights)  # Normalize to sum to 1
# comp_2_weights = np.abs(eigenvectors[:, 1]) * eigenvalues[1]
# comp_2_weights = comp_2_weights / np.sum(comp_2_weights)  # Normalize to sum to 1
# comp_3_weights = np.abs(eigenvectors[:, 2]) * eigenvalues[2]
# comp_3_weights = comp_3_weights / np.sum(comp_3_weights)  # Normalize to sum to 1
# comp_4_weights = np.abs(eigenvectors[:, 3]) * eigenvalues[3]
# comp_4_weights = comp_4_weights / np.sum(comp_4_weights)  # Normalize to sum to 1
# comp_5_weights = np.abs(eigenvectors[:, 4]) * eigenvalues[4]
# comp_5_weights = comp_5_weights / np.sum(comp_5_weights)  # Normalize to sum to 1

# print("\nComponent 1 weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, comp_1_weights):
#     print(f"{stat}: {weight:.4f}")
# print("\nComponent 2 weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, comp_2_weights):
#     print(f"{stat}: {weight:.4f}")
# print("\nComponent 3 weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, comp_3_weights):
#     print(f"{stat}: {weight:.4f}")
# print("\nComponent 4 weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, comp_4_weights):
#     print(f"{stat}: {weight:.4f}")
# print("\nComponent 5 weights (normalized to sum to 1):")
# for stat, weight in zip(stat_names, comp_5_weights):
#     print(f"{stat}: {weight:.4f}")

In [239]:
# Outputting the final weights to a json file in the project root called cm_weights.json
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
weights_dict = dict(zip(col_names, final_weights))
weights_path = project_root / "workshop" / "cm_weights.json"
with open(weights_path, "w") as f:
    json.dump(weights_dict, f, indent=4)

In [240]:
# Now lets look at storing the means and standard deviations used for the z-score calculations when
# calculating PCA, in a json file called cm_means_stds.json in the project root, with the following format:
# {
#     "goals": {"mean": 0.1, "std": 0.5},
#    ...
# }

means_stds = {}
for col in col_names:
    # if col == "goals":
    #     mean = np.log(cm_players_df["team_xg"] * cm_goal_share + 1)
    #     std = np.log(cm_players_df[col].std())
    # elif col == "assists":
    #     mean = np.log(cm_players_df["team_xg"] * cm_assist_share + 1)
    #     std = np.log(cm_players_df[col].std())
    # else:
    #     mean = cm_players_df[col].mean()
    #     std = cm_players_df[col].std()
    mean = cm_players_df[col].mean()
    std = cm_players_df[col].std()
    means_stds[col] = {"mean": mean, "std": std}
means_stds_path = project_root / "workshop" / "cm_means_stds.json"
with open(means_stds_path, "w") as f:
    json.dump(means_stds, f, indent=4)